# 03 · News — vector RAG (semantic, filtered, hybrid, cited)

`AgensgraphVectorStore` as a LlamaIndex `VectorStoreIndex` over news articles.
This notebook tours four retrieval modes.

> Run `ingest.py` first to build the `news` graph.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

configure_settings()
from llama_index.core import VectorStoreIndex
store = agens.make_vector_store(graph_name="news", node_label="Article")
index = VectorStoreIndex.from_vector_store(store, embed_model=get_embed_model())
question = "What are companies doing with artificial intelligence?"

## (a) Plain semantic search

HNSW nearest-neighbour over the chunk embeddings.

In [2]:
def show(hits):
    for h in hits:
        m = h.node.metadata or {}
        print(f"{h.score:.3f}  {m.get('domain','?')} · {m.get('date','?')} · {(m.get('title') or '')[:55]}")
show(index.as_retriever(similarity_top_k=5).retrieve(question))

0.650  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.637  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.621  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.620  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.619  www.techrepublic.com · 2017-12-11 · Why some of the world's biggest companies are using AI 


## (b) Metadata-filtered retrieval

`MetadataFilters` translate to an indexed Cypher `WHERE` — here `domain IN [...]`
`AND date >= 2017-01-01`.

In [3]:
from llama_index.core.vector_stores import (MetadataFilters, MetadataFilter,
                                             FilterOperator, FilterCondition)
domains = [r["domain"] for r in store.database_query(
    'MATCH (n:"Article") WHERE n.domain IS NOT NULL '
    'RETURN n.domain AS domain, count(*) AS c ORDER BY c DESC LIMIT 4')]
filters = MetadataFilters(condition=FilterCondition.AND, filters=[
    MetadataFilter(key="domain", operator=FilterOperator.IN, value=domains),
    MetadataFilter(key="date", operator=FilterOperator.GTE, value="2017-01-01")])
print("domains:", domains)
show(index.as_retriever(similarity_top_k=5, filters=filters).retrieve(question))

domains: ['nationalpost.com', 'www.taiwannews.com.tw', 'www.nigeriatoday.ng', 'abcnews.go.com']


0.541  www.taiwannews.com.tw · 2018-04-24 · Splunk Customers Accelerate Business Value Through Arti
0.538  www.taiwannews.com.tw · 2018-04-25 · Global Artificial Intelligence Market in Education Sect
0.534  www.taiwannews.com.tw · 2018-04-25 · Global Artificial Intelligence Market in Education Sect
0.522  www.taiwannews.com.tw · 2018-04-24 · Splunk Customers Accelerate Business Value Through Arti


## (c) Hybrid search (vector + keyword RRF)

A `hybrid_search=True` store fuses HNSW semantic search with full-text keyword
search by reciprocal rank fusion (each modality uses its own index).

In [4]:
hstore = agens.make_vector_store(graph_name="news", node_label="Article", hybrid_search=True)
hindex = VectorStoreIndex.from_vector_store(hstore, embed_model=get_embed_model())
show(hindex.as_retriever(similarity_top_k=5, vector_store_query_mode="hybrid").retrieve(question))

0.016  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.016  nationalpost.com · 2018-04-24 · Aeroplan's troublesome 'purity of the country' survey i
0.016  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.016  uproxx.com · 2018-04-24 · China Plans To Implement Social Credit To 1.4 Billion C
0.016  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI


## (d) Cited RAG

`CitationQueryEngine` answers with inline `[N]` markers tied to sources.

In [5]:
from llama_index.core.query_engine import CitationQueryEngine
resp = CitationQueryEngine.from_args(index, similarity_top_k=5).query(question)
print(str(resp).strip())
print("\nSources:")
for i, s in enumerate(resp.source_nodes, 1):
    m = s.node.metadata or {}
    print(f"  [{i}] {m.get('domain','?')} · {(m.get('title') or '')[:55]}")

Companies are leveraging artificial intelligence (AI) in various ways to enhance their operations and improve efficiency. For instance, AI is being used to automate the scanning of websites for suspicious content to help stop predators [1]. In the recruitment process, companies like Unilever utilize AI to analyze candidates' responses, body language, and tone, which streamlines hiring and increases acceptance rates [1]. Additionally, AI assists in customer service by processing natural language and routing inquiries to the appropriate agents, thereby enhancing the support provided to customers [1]. 

Moreover, businesses are optimistic about AI's potential to create net job gains, with many executives believing that intelligent technology will be critical for market differentiation [2][3]. However, there is a noted disconnect in investment for reskilling workers to adapt to these changes, as only a small percentage of leaders plan to significantly increase such investments in the near 

## How it was built

`ingest.py` chunks CC-News, embeds in parallel, and `async_add`s with metadata;
it indexes the filterable keys so filtered search stays fast:

```python
nodes = SentenceSplitter(chunk_size=256).get_nodes_from_documents(docs)
await store.async_add(nodes)            # nodes carry {domain, date, title, url}
store.create_property_index("domain"); store.create_property_index("date")
```

In [6]:
agens.close()